#  Data Cleaning

This notebook performs data cleaning on a synthetic banking dataset containing customers, accounts, transactions, cards, loans, branches, merchants, dates, and complaints. The objective is to prepare clean, consistent, and analysis-ready datasets for Exploratory Data Analysis (EDA) and dashboard development.

### Cleaning tasks performed
- Loaded all raw CSV files.
- Checked dataset size and structure.
- Removed duplicate records.
- Handled missing values.
- Standardized text values.
- Verified data types.
- Performed basic outlier inspection.
- Validated the cleaned data.
- Saved cleaned datasets for the next phase.


## Import Libraries

In [20]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

## Loading the Datasets

In [21]:
customers = pd.read_csv("data/Raw/dim_customers.csv")
accounts = pd.read_csv("data/Raw/dim_accounts.csv")
cards = pd.read_csv("data/Raw/dim_cards.csv")
branches = pd.read_csv("data/Raw/dim_branches.csv")
merchants = pd.read_csv("data/Raw/dim_merchants.csv")
dates = pd.read_csv("data/Raw/dim_date.csv")
transactions = pd.read_csv("data/Raw/fact_transactions.csv")
loans = pd.read_csv("data/Raw/fact_loans.csv")
complaints = pd.read_csv("data/Raw/fact_complaints.csv")

##  Store Datasets

In [22]:
datasets = {
    "Customers": customers,
    "Accounts": accounts,
    "Cards": cards,
    "Branches": branches,
    "Merchants": merchants,
    "Date": dates,
    "Transactions": transactions,
    "Loans": loans,
    "Complaints": complaints
}

## Check Primary Key Uniqueness
Validating that there are no duplicate primary keys in dimension tables.

In [23]:
print('Duplicates in Customers:', customers['customer_id'].duplicated().sum())
print('Duplicates in Accounts:', accounts['account_id'].duplicated().sum())
print('Duplicates in Cards:', cards['card_id'].duplicated().sum())
print('Duplicates in Branches:', branches['branch_id'].duplicated().sum())
print('Duplicates in Merchants:', merchants['merchant_id'].duplicated().sum())


Duplicates in Customers: 0
Duplicates in Accounts: 0
Duplicates in Cards: 0
Duplicates in Branches: 0
Duplicates in Merchants: 0


## Check Numeric Outliers
We will inspect numeric distributions using `.describe()` to see if there are impossible values (like negative age). We won't remove them unless truly impossible.

In [24]:
display(transactions['amount'].describe())
display(customers[['age', 'monthly_income', 'credit_score']].describe())


count    2.522773e+06
mean     5.381816e+04
std      2.521448e+05
min     -1.970345e+07
25%      2.424800e+03
50%      2.356853e+04
75%      6.266944e+04
max      2.678984e+07
Name: amount, dtype: float64

,age,monthly_income,credit_score
count,150000.000000,150000.000000,150000.000000
mean,39.749800,80736.424227,682.863553
std,12.474393,76454.115137,90.997108
min,18.000000,12000.000000,349.000000
25%,31.000000,32951.000000,618.000000
50%,39.000000,56703.500000,680.000000
75%,48.000000,97349.500000,747.000000
max,85.000000,399999.000000,900.000000


## Dataset Size Before Cleaning

In [25]:
before_cleaning = pd.DataFrame({

    "Table": datasets.keys(),

    "Rows":[len(df) for df in datasets.values()]
})

before_cleaning

,Table,Rows
0,Customers,150000
1,Accounts,240000
2,Cards,300000
3,Branches,25
4,Merchants,3500
5,Date,1096
6,Transactions,2522773
7,Loans,120000
8,Complaints,11124


## Remove Duplicate Records

During the Data Understanding phase, duplicate rows were identified only in the Transactions table.

No duplicate rows were found in the remaining tables, so only the Transactions table requires cleaning.

In [26]:
# Number of rows before removing duplicates
before_rows = len(transactions)

# Remove duplicate rows
transactions = transactions.drop_duplicates()
datasets["Transactions"] = transactions

# Number of rows after removing duplicates
after_rows = len(transactions)


print("Rows before removing duplicates:", before_rows)
print("Rows after removing duplicates :", after_rows)
print("Duplicate rows removed          :", before_rows - after_rows)

Rows before removing duplicates: 2522773
Rows after removing duplicates : 2499401
Duplicate rows removed          : 23372


## Check Duplicate Transaction IDs

Removing duplicate rows only catches rows that are identical in every column. It is also important to check if `transaction_id` itself repeats, since each transaction should have a unique ID.


In [27]:
print('Duplicate transaction_id before fix:', transactions['transaction_id'].duplicated().sum())

transactions = transactions.drop_duplicates(subset='transaction_id', keep='first')
datasets['Transactions'] = transactions

print('Duplicate transaction_id after fix :', transactions['transaction_id'].duplicated().sum())
print('Total transaction rows now         :', len(transactions))

Duplicate transaction_id before fix: 1606
Duplicate transaction_id after fix : 0
Total transaction rows now         : 2497795


## Dataset Size After Cleaning

In [28]:
after_cleaning = pd.DataFrame({
    "Table": datasets.keys(),
    "Rows": [len(df) for df in datasets.values()]
})

after_cleaning

,Table,Rows
0,Customers,150000
1,Accounts,240000
2,Cards,300000
3,Branches,25
4,Merchants,3500
5,Date,1096
6,Transactions,2497795
7,Loans,120000
8,Complaints,11124


### Observation

- Duplicate records were found only in the **Transactions** table.
- Full duplicate rows were removed using `drop_duplicates()`.
- A separate check on `transaction_id` showed that some IDs were still repeating even after that step, so those were removed as well using `subset='transaction_id'`.
- No duplicate records were found in the remaining datasets.
- This ensures each banking transaction is counted only once during analysis.


##  Handle Missing Values

In [29]:
customers[
    ["gender","occupation","education","income_bracket"]
].isnull().sum()

gender            3000
occupation        3000
education         2250
income_bracket    4500
dtype: int64

In [30]:
customers["gender"] = customers["gender"].fillna("Unknown")

customers["occupation"] = customers["occupation"].fillna("Unknown")

customers["education"] = customers["education"].fillna("Unknown")

customers["income_bracket"] = customers["income_bracket"].fillna("Unknown")

### Observation

Missing values were identified in customer-related categorical columns before cleaning.


Columns cleaned:
- gender
- occupation
- education
- income_bracket

Missing values were replaced with **"Unknown"** instead of deleting records so valuable customer information was not lost.




##  Standardize Text

In [31]:
customers["gender"] = customers["gender"].str.title()

transactions["status"] = transactions["status"].str.title()

complaints["status"] = complaints["status"].str.title()

loans["loan_status"] = loans["loan_status"].str.title()

### Observation

Text values were standardized using title case to keep categories consistent.

Example:
- male → Male
- FEMALE → Female
- completed → Completed

This prevents duplicate categories during EDA and visualization.


## Convert Date Columns



In [32]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

accounts["open_date"] = pd.to_datetime(accounts["open_date"])

cards["issue_date"] = pd.to_datetime(cards["issue_date"])

loans["disbursement_date"] = pd.to_datetime(loans["disbursement_date"])

dates["full_date"] = pd.to_datetime(dates["full_date"])

## Business-Related Missing Values

Some missing values were intentionally preserved because they represent valid business scenarios rather than data quality issues.

Examples:

- `card_id` is missing for transactions that do not use a debit or credit card (such as salary credits or bank transfers).
- `merchant_id` is missing for transactions that do not involve a merchant.
- `linked_transaction_id` is only available for complaints related to specific transactions.
- `resolution_days` is missing for complaints that are still Open, In Progress, or Escalated.

These values were not modified to preserve the business meaning of the data.

## Data Validation After Cleaning

In [33]:
missing_after = []

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    for column, value in missing.items():

        missing_after.append({

            "Table":name,

            "Column":column,

            "Missing Values":value
        })

missing_after = pd.DataFrame(missing_after)

missing_after

,Table,Column,Missing Values
0,Transactions,card_id,1297795
1,Transactions,merchant_id,37442
2,Transactions,channel,24991
3,Complaints,linked_transaction_id,9000
4,Complaints,resolution_days,3326


In [34]:
duplicate_summary = pd.DataFrame({

    "Table":datasets.keys(),

    "Duplicate Rows":[

        customers.duplicated().sum(),

        accounts.duplicated().sum(),

        cards.duplicated().sum(),

        branches.duplicated().sum(),

        merchants.duplicated().sum(),

        dates.duplicated().sum(),

        transactions.duplicated().sum(),

        loans.duplicated().sum(),

        complaints.duplicated().sum()
    ]
})

duplicate_summary

,Table,Duplicate Rows
0,Customers,0
1,Accounts,0
2,Cards,0
3,Branches,0
4,Merchants,0
5,Date,0
6,Transactions,0
7,Loans,0
8,Complaints,0


In [35]:
print('Duplicate transaction_id after final cleaning:', transactions['transaction_id'].duplicated().sum())

Duplicate transaction_id after final cleaning: 0


##  Outlier Inspection

We'll use simple boxplots and describe(), not advanced statistics.

In [36]:
transactions["amount"].describe()

count    2.497795e+06
mean     5.380280e+04
std      2.516229e+05
min     -1.970345e+07
25%      2.424580e+03
50%      2.356837e+04
75%      6.267147e+04
max      2.678984e+07
Name: amount, dtype: float64

In [37]:
transactions.boxplot(column="amount")

<Axes: >

### Observation

The transaction amount column was inspected using descriptive statistics and a boxplot.

Some very high and negative transaction amounts were observed. These values were not removed because they may represent valid banking activities such as withdrawals, refunds, transfers, or high-value transactions rather than data quality issues.

Outlier treatment will be performed during the EDA phase only if required for a specific analysis.

##  Save Clean Dataset

This allows future analysis to use clean and consistent data instead of the original raw files.


In [38]:
customers.to_csv("data/Cleaned/dim_customers.csv", index=False)

accounts.to_csv("data/Cleaned/dim_accounts.csv", index=False)

cards.to_csv("data/Cleaned/dim_cards.csv", index=False)

branches.to_csv("data/Cleaned/dim_branches.csv", index=False)

merchants.to_csv("data/Cleaned/dim_merchants.csv", index=False)

dates.to_csv("data/Cleaned/dim_date.csv", index=False)

transactions.to_csv("data/Cleaned/fact_transactions.csv", index=False)

loans.to_csv("data/Cleaned/fact_loans.csv", index=False)

complaints.to_csv("data/Cleaned/fact_complaints.csv", index=False)

print("Clean datasets exported successfully.")

Clean datasets exported successfully.


# Data Cleaning Summary

## Work Completed

- Loaded all nine banking datasets.
- Removed duplicate transaction records.
- Filled missing values in customer demographic columns.
- Standardized text formatting for consistency.
- Converted date columns to the correct data type.
- Inspected potential outliers in numerical data.
- Validated primary keys and relationships between tables.
- Preserved business-related missing values where appropriate.
- Exported the cleaned datasets for further analysis.

## Result

The datasets are clean, consistent, and ready for Exploratory Data Analysis (EDA), visualization, and dashboard creation.